In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
import asyncio
import time
import json
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_groq import ChatGroq
from langchain_mistralai import ChatMistralAI

# ---------- Schema ----------
class ReviewAnalysis(BaseModel):
    sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="Overall sentiment of the review"
    )
    key_issues: list[str] = Field(
        description="Specific problems, complaints, or standout praise mentioned. Empty list if none."
    )
    summary: str = Field(
        description="One-sentence, neutral summary of the review's content"
    )

In [15]:
# ---------- Rate limiter (adjust requests_per_second to your plan's limit) ----------
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0,   # 1 request every 2 seconds
    check_every_n_seconds=0.1,
    max_bucket_size=1,
)

# ---------- Models ----------
groq_model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, rate_limiter=rate_limiter)
mistral_model = ChatMistralAI(model="mistral-large-latest", temperature=0, rate_limiter=rate_limiter)

prompt = ChatPromptTemplate.from_template(
    "Analyze the following product review.\n\nReview: {review}"
)

groq_chain = prompt | groq_model.with_structured_output(ReviewAnalysis)
mistral_chain = prompt | mistral_model.with_structured_output(ReviewAnalysis)

In [13]:
# ---------- Sample data ----------
reviews = [
    "Battery dies in 2 hours, but the screen is gorgeous.",
    "Customer support never responded to my three emails. Very frustrating.",
    "Works exactly as described. No complaints at all.",
    "Shipping took 3 weeks and the box arrived damaged.",
    "Love the design, but the app crashes constantly.",
    "Best purchase I've made this year, highly recommend.",
    "Overpriced for what you get. Build quality feels cheap.",
    "Setup was confusing but once configured, it runs great.",
    "Cancelled my order because the price jumped at checkout.",
    "Decent product, nothing special, does the job.",
]

# ---------- Sequential (ainvoke) ----------
async def run_sequential(chain, reviews, label):
    outputs = []
    start = time.time()
    for r in reviews:
        result = await chain.ainvoke({"review": r})
        outputs.append(result.model_dump())
    elapsed = time.time() - start
    print(f"[{label}] Sequential (ainvoke): {elapsed:.2f}s for {len(reviews)} reviews")
    return outputs, elapsed

# ---------- Concurrent (abatch) ----------
async def run_abatch(chain, reviews, label, max_concurrency=3):
    inputs = [{"review": r} for r in reviews]
    start = time.time()
    results = await chain.abatch(inputs, config={"max_concurrency": max_concurrency})
    elapsed = time.time() - start
    outputs = [r.model_dump() for r in results]
    print(f"[{label}] Concurrent (abatch): {elapsed:.2f}s for {len(reviews)} reviews")
    return outputs, elapsed

In [ ]:
# ---------- Run comparison ----------
results = {}

for label, chain in [("groq", groq_chain), ("mistral", mistral_chain)]:
    seq_out, seq_time = await run_sequential(chain, reviews, label)
    batch_out, batch_time = await run_abatch(chain, reviews, label)

    results[label] = {"sequential": seq_time, "abatch": batch_time}

    with open(f"reviews_{label}_sequential.json", "w") as f:
        json.dump(seq_out, f, indent=2)
    with open(f"reviews_{label}_abatch.json", "w") as f:
        json.dump(batch_out, f, indent=2)

# ---------- Summary ----------
print(f"\n{'Provider':<10} {'Sequential':<12} {'abatch':<10}")
for label, t in results.items():
    print(f"{label:<10} {t['sequential']:<12.2f} {t['abatch']:<10.2f}")